# 02 · Lead intelligence
**Objective:** compare the rule baseline with a real model at ICP classification on the *train* split.
Held-out numbers belong in notebook 06; this notebook is for exploring, so tuning here does not contaminate them.

Data is fictional. Labels come from a known rule, not real sales outcomes.

In [ ]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userSecrets
    try:
        os.environ["HF_TOKEN"] = userSecrets.get("HF_TOKEN")
    except userSecrets.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

from app.agents.llm import LLMError, ModelRegistry

registry = ModelRegistry()
status = {s.key: s for s in registry.status()}
CANDIDATES = ["qwen3-4b", "qwen2.5-7b", "mistral-7b", "hf-qwen3-8b"]
MODEL = next((k for k in CANDIDATES if status[k].available), None)
if MODEL is None:
    # There is deliberately no mock: results must come from a real model.
    raise RuntimeError("No model is reachable. Start Ollama and pull a model, or set HF_TOKEN.\n"
                       + "\n".join(f"{s.key}: {s.reason}" for s in status.values() if not s.available))
print(f"Using real model: {MODEL} ({registry.model(MODEL).model})")

In [ ]:
from app.agents.evaluation import baseline, metrics, tasks
from app.agents.evaluation.runner import Dataset

data = Dataset.load("train")
icp = data.manifest["icp"]
sample = data.prospects[:8]  # Increase for a fuller picture; each item is a real model call.
print(sample[0]["structured"]); print(sample[0]["note"]); print("gold:", sample[0]["gold"]["icp_fit"])

In [ ]:
gold, rule_pred, model_pred, reasons = [], [], [], []
for r in sample:
    gold.append(r["gold"]["icp_fit"])
    rule_pred.append(baseline.classify(r, icp).fit)
    try:
        out, _ = registry.structured(MODEL, tasks.classification_messages(r, icp), tasks.ClassificationOut, max_tokens=400)
        model_pred.append(out.fit); reasons.append(out.reasons)
    except LLMError as exc:  # A failed call counts as wrong; it is not hidden.
        model_pred.append(None); reasons.append([f"{exc.category}: {exc}"])

for name, pred in (("rules", rule_pred), (MODEL, model_pred)):
    report = metrics.classification_report(gold, pred)
    low, high = report["accuracy_ci95"]
    print(f"{name:12} accuracy {report['accuracy']:.0%} (95% CI {low:.0%}-{high:.0%}, n={report['n']})")

## Why the model decided
Reasons are shown so a person can check them against the note.

In [ ]:
for r, p, why in zip(sample, model_pred, reasons):
    print(f"{r['id']}  gold={r['gold']['icp_fit']:6} model={p}  {why[:2]}")
assert len(model_pred) == len(sample)

**Expected:** rules score very well here because their keywords were written from this split. Compare on held-out
data in notebook 06 before concluding anything.

**Limitations:** eight items give a very wide interval. A label is an assessment of fit, never a conversion probability.